In [15]:
import requests
import pandas as pd
import os
from collections import defaultdict
import time
import json
import datetime
import asyncio
import types
import nest_asyncio
nest_asyncio.apply()  

from fastf1.livetiming.client import SignalRClient

lap_times = {}

def fix_json(line):
    # Fix F1's not JSON compliant data
    line = line.replace("'", '"') \
        .replace('True', 'true') \
        .replace('False', 'false')
    return line

def _handle_message_overwrite(self, msg):
    """Custom message handler for the SignalRClient"""
    msg = fix_json(msg)
    try:
        cat, msg, dt = json.loads(msg)
        if cat == "TimingData":
            process_timing_data(msg)
    except (json.JSONDecodeError, ValueError):
        raise ValueError(f"Invalid JSON message: {msg}")
    
def _start_overwrite(self):
    """Connect to the data stream and start collecting data"""
    try:
        # Instead of asyncio.run(), use the current event loop
        loop = asyncio.get_event_loop()
        loop.run_until_complete(self._async_start())
    except KeyboardInterrupt:
        raise KeyboardInterrupt
    
def process_timing_data(msg):
    """Process timing data to extract lap times"""
    try:
        data = json.loads(msg)
        
        # Extract lap time information
        if isinstance(data, dict):
            if "Lines" in data:
                for driver_code, driver_data in data["Lines"].items():
                    if "Sectors" in driver_data and "Speeds" in driver_data:
                        # This looks like lap time data
                        if "LastLapTime" in driver_data:
                            lap_time = driver_data["LastLapTime"]["Value"]
                            lap_number = driver_data.get("NumberOfLaps", 0)
                            
                            # Store lap time
                            if driver_code not in lap_times:
                                lap_times[driver_code] = []
                            
                            lap_times[driver_code].append({
                                "Lap": lap_number,
                                "LapTime": lap_time,
                                "Timestamp": datetime.datetime.now().isoformat()
                            })
                            
                            # Print instead of raising an exception
                            print(f"Driver {driver_code} completed lap {lap_number} with time {lap_time}")
    except Exception as e:
        print(f"Error processing timing data: {e}")

parent_dir = os.getcwd()
#parent_dir = os.path.dirname(os.path.abspath(__file__))
#data_dir = os.path.join(parent_dir, '2025_data')
#results = pd.read_csv(os.path.join(data_dir, 'results_with_elo_with_championships.csv'))
#last_race_number = results['RoundNumber'].max()
#last_race = results[results['RoundNumber'] == last_race_number]
#results_last_race = last_race.drop(columns=['elo_before', 'DriverWinsCount_before', 'DriverWinsCount_before', 'TeamPointsBefore', 'DriverWinsCount_before'])

client = SignalRClient("unused.txt")
client.topics = ['TimingData']
client._handle_message = types.MethodType(_handle_message_overwrite, client)
client.start = types.MethodType(_start_overwrite, client)
client.start()




2025-06-09 18:57:54,063 - INFO: Starting FastF1 live timing client [v3.5.3]


KeyboardInterrupt: 

2025-06-09 18:58:54,140 - WARNING: Timeout - received no data for more than 60 seconds!
